In [11]:
from dataclasses import dataclass
import os
import oqs
import pytest

## ML_KEM 
Shared secret for session (ss)

In [12]:
'''
Key encapsulation mechanism 
ML - KEM - 768 protocol (Kyber)

1. Key Generation  (pk,sk) = Gen()
2. Encapsulation    (k,c) = Encap(pk)
3. Decapsulation    k' = Decap(sk,c')

'''


KEM_Algo = "ML-KEM-768"


KEM_ALGORITHM = "ML-KEM-768"


# Data structure 

@dataclass
class KEMKeyPair:
    """
    Paire de clés ML-KEM-768.

    Tailles FIPS 203 :
      - public_key  : 1184 octets
      - secret_key  : 2400 octets
    """
    public_key: bytes
    secret_key: bytes

    def public_key_hex(self) -> str:
        return self.public_key.hex()

    def __repr__(self):
        return (
            f"KEMKeyPair("
            f"pk[{len(self.public_key)}B]={self.public_key.hex()[:16]}…, "
            f"sk[{len(self.secret_key)}B]=***)"
        )


@dataclass
class KEMEncapsulationResult:
    """
    Résultat d'une encapsulation ML-KEM.

    - ciphertext    : à envoyer à l'autre partie (1088 octets)
    - shared_secret : secret gardé localement    (32 octets)
    """
    ciphertext: bytes
    shared_secret: bytes

    def __repr__(self):
        return (
            f"KEMEncapsulationResult("
            f"ct[{len(self.ciphertext)}B]={self.ciphertext.hex()[:16]}…, "
            f"ss={self.shared_secret.hex()})"
        )


# ---------------------------------------------------------------------------
# Fonctions principales
# ---------------------------------------------------------------------------

def generate_keypair() -> KEMKeyPair:
    """
    Génère une paire de clés ML-KEM-768.

    Utilisé par :
      - Bob pour sa clé d'identité (IK)
      - Bob pour sa Signed PreKey (SPK)
      - Bob pour ses One-Time PreKeys (OPK)

    Returns:
        KEMKeyPair(public_key, secret_key)
    """
    with oqs.KeyEncapsulation(KEM_ALGORITHM) as kem:
        public_key = kem.generate_keypair()
        secret_key = kem.export_secret_key()

    return KEMKeyPair(public_key=public_key, secret_key=secret_key)


def encapsulate(public_key: bytes) -> KEMEncapsulationResult:
    """
    Encapsule un secret partagé avec la clé publique du destinataire.
    Appelé par l'initiateur (Alice) durant le handshake PQ-X3DH.

    Args:
        public_key : clé publique du destinataire (Bob)

    Returns:
        KEMEncapsulationResult :
          - ciphertext    → à envoyer à Bob via le serveur
          - shared_secret → conservé par Alice pour dériver la master key

    Raises:
        ValueError : si la clé publique a une taille incorrecte
        RuntimeError : si liboqs échoue
    """
    _validate_public_key(public_key)

    with oqs.KeyEncapsulation(KEM_ALGORITHM) as kem:
        ciphertext, shared_secret = kem.encap_secret(public_key)

    return KEMEncapsulationResult(ciphertext=ciphertext, shared_secret=shared_secret)


def decapsulate(secret_key: bytes, ciphertext: bytes) -> bytes:
    """
    Décapsule le secret partagé avec la clé secrète locale.
    Appelé par le destinataire (Bob) durant le handshake PQ-X3DH
    et à chaque tour du KEM Ratchet.

    Args:
        secret_key  : clé secrète de Bob
        ciphertext  : ciphertext reçu d'Alice

    Returns:
        shared_secret (32 octets) — identique au shared_secret d'Alice
        si les clés sont correctes.

    Note IND-CCA2 :
        Si le ciphertext est modifié (attaque active), ML-KEM ne lève
        pas d'exception — il retourne un secret pseudo-aléatoire incorrect.
        C'est un comportement intentionnel (pas d'oracle d'erreur).

    Raises:
        ValueError : si les tailles sont incorrectes
        RuntimeError : si liboqs échoue
    """
    _validate_secret_key(secret_key)
    _validate_ciphertext(ciphertext)

    with oqs.KeyEncapsulation(KEM_ALGORITHM, secret_key=secret_key) as kem:
        shared_secret = kem.decap_secret(ciphertext)

    return shared_secret


# ---------------------------------------------------------------------------
# KEM Ratchet — utilisé dans double_ratchet.py
# ---------------------------------------------------------------------------

def kem_ratchet_step(their_public_key: bytes) -> tuple[bytes, bytes]:
    """
    Effectue un pas du KEM Ratchet (remplace le DH Ratchet de Signal).

    Appelé par l'émetteur à chaque nouveau "tour" de ratchet.

    Returns:
        (our_new_public_key, shared_secret)
        - our_new_public_key : à inclure dans le header du message
        - shared_secret      : utilisé pour dériver les nouvelles clés de chaîne

    Usage dans double_ratchet.py :
        new_pk, ratchet_secret = kem_ratchet_step(their_ratchet_public_key)
    """
    result = encapsulate(their_public_key)
    # On génère aussi notre nouvelle clé pour le prochain tour
    new_keypair = generate_keypair()
    return new_keypair.public_key, result.shared_secret


# ---------------------------------------------------------------------------
# Validation interne
# ---------------------------------------------------------------------------

def _validate_public_key(public_key: bytes) -> None:
    info = kem_info()
    expected = info["public_key_bytes"]
    if len(public_key) != expected:
        raise ValueError(
            f"Taille de clé publique incorrecte : "
            f"reçu {len(public_key)} octets, attendu {expected}"
        )


def _validate_secret_key(secret_key: bytes) -> None:
    info = kem_info()
    expected = info["secret_key_bytes"]
    if len(secret_key) != expected:
        raise ValueError(
            f"Taille de clé secrète incorrecte : "
            f"reçu {len(secret_key)} octets, attendu {expected}"
        )


def _validate_ciphertext(ciphertext: bytes) -> None:
    info = kem_info()
    expected = info["ciphertext_bytes"]
    if len(ciphertext) != expected:
        raise ValueError(
            f"Taille de ciphertext incorrecte : "
            f"reçu {len(ciphertext)} octets, attendu {expected}"
        )


# ---------------------------------------------------------------------------
# Utilitaires
# ---------------------------------------------------------------------------

def kem_info() -> dict:
    """Retourne les paramètres de taille de ML-KEM-768."""
    with oqs.KeyEncapsulation(KEM_ALGORITHM) as kem:
        d = kem.details
    return {
        "algorithm":             d["name"],
        "claimed_nist_level":    d["claimed_nist_level"],
        "public_key_bytes":      d["length_public_key"],
        "secret_key_bytes":      d["length_secret_key"],
        "ciphertext_bytes":      d["length_ciphertext"],
        "shared_secret_bytes":   d["length_shared_secret"],
    }


def is_available() -> bool:
    """Vérifie que ML-KEM-768 est disponible dans liboqs."""
    return KEM_ALGORITHM in oqs.get_enabled_kem_mechanisms()


### KEM Test

In [13]:


# Tailles FIPS 203 — ML-KEM-768
PK_SIZE = 1184
SK_SIZE = 2400
CT_SIZE = 1088
SS_SIZE = 32


# Availability

class TestAvailability:
    def test_ml_kem_768_is_available(self):
        assert is_available(), f"{KEM_ALGORITHM} non disponible dans liboqs"

    def test_kem_info_returns_correct_sizes(self):
        info = kem_info()
        assert info["public_key_bytes"] == PK_SIZE
        assert info["secret_key_bytes"] == SK_SIZE
        assert info["ciphertext_bytes"] == CT_SIZE
        assert info["shared_secret_bytes"] == SS_SIZE
        assert info["claimed_nist_level"] == 3

# Key generation


class TestGenerateKeypair:
    def test_returns_keypair_type(self):
        assert isinstance(generate_keypair(), KEMKeyPair)

    def test_correct_sizes(self):
        kp = generate_keypair()
        assert len(kp.public_key) == PK_SIZE
        assert len(kp.secret_key) == SK_SIZE

    def test_keys_are_not_zero(self):
        kp = generate_keypair()
        assert kp.public_key != bytes(PK_SIZE)
        assert kp.secret_key != bytes(SK_SIZE)

    def test_each_keypair_is_unique(self):
        kp1 = generate_keypair()
        kp2 = generate_keypair()
        assert kp1.public_key != kp2.public_key
        assert kp1.secret_key != kp2.secret_key

    def test_public_key_hex(self):
        kp = generate_keypair()
        hex_str = kp.public_key_hex()
        assert isinstance(hex_str, str)
        assert len(hex_str) == PK_SIZE * 2


# Encapsulation

class TestEncapsulate:
    @pytest.fixture
    def bob(self):
        return generate_keypair()

    def test_returns_result_type(self, bob):
        assert isinstance(encapsulate(bob.public_key), KEMEncapsulationResult)

    def test_correct_sizes(self, bob):
        result = encapsulate(bob.public_key)
        assert len(result.ciphertext) == CT_SIZE
        assert len(result.shared_secret) == SS_SIZE

    def test_shared_secret_not_zero(self, bob):
        result = encapsulate(bob.public_key)
        assert result.shared_secret != bytes(SS_SIZE)

    def test_probabilistic_each_call_differs(self, bob):
        """Deux encapsulations sur la même clé → ciphertexts et secrets différents."""
        r1 = encapsulate(bob.public_key)
        r2 = encapsulate(bob.public_key)
        assert r1.ciphertext != r2.ciphertext
        assert r1.shared_secret != r2.shared_secret

    def test_invalid_key_size_raises_value_error(self):
        with pytest.raises(ValueError, match="Taille de clé publique incorrecte"):
            encapsulate(b"trop_court")

    def test_empty_key_raises_value_error(self):
        with pytest.raises(ValueError):
            encapsulate(b"")


# Decapslation

class TestDecapsulate:
    @pytest.fixture
    def bob(self):
        return generate_keypair()

    def test_returns_bytes(self, bob):
        result = encapsulate(bob.public_key)
        ss = decapsulate(bob.secret_key, result.ciphertext)
        assert isinstance(ss, bytes)

    def test_correct_size(self, bob):
        result = encapsulate(bob.public_key)
        ss = decapsulate(bob.secret_key, result.ciphertext)
        assert len(ss) == SS_SIZE

    def test_invalid_secret_key_raises_value_error(self, bob):
        result = encapsulate(bob.public_key)
        with pytest.raises(ValueError, match="Taille de clé secrète incorrecte"):
            decapsulate(b"trop_court", result.ciphertext)

    def test_invalid_ciphertext_raises_value_error(self, bob):
        with pytest.raises(ValueError, match="Taille de ciphertext incorrecte"):
            decapsulate(bob.secret_key, b"trop_court")


# Main handshake


class TestAliceBobHandshake:
    """
    Simule le handshake de base du protocole :
    Bob publie sa clé, Alice encapsule, Bob décapsule.
    Les deux doivent obtenir le même secret.
    """

    def test_shared_secrets_match(self):
        bob = generate_keypair()
        alice_result = encapsulate(bob.public_key)
        bob_secret = decapsulate(bob.secret_key, alice_result.ciphertext)
        assert alice_result.shared_secret == bob_secret

    def test_five_independent_sessions_all_match(self):
        """Simule 5 sessions indépendantes."""
        bob = generate_keypair()
        for i in range(5):
            alice_result = encapsulate(bob.public_key)
            bob_secret = decapsulate(bob.secret_key, alice_result.ciphertext)
            assert alice_result.shared_secret == bob_secret, f"Session {i} échouée"

    def test_five_sessions_all_have_different_secrets(self):
        """Chaque session produit un secret distinct."""
        bob = generate_keypair()
        secrets = [encapsulate(bob.public_key).shared_secret for _ in range(5)]
        assert len(set(secrets)) == 5

    def test_wrong_secret_key_gives_wrong_secret(self):
        """
        Eve ne peut pas décapsuler le message d'Alice destiné à Bob.
        """
        bob = generate_keypair()
        eve = generate_keypair()
        alice_result = encapsulate(bob.public_key)
        eve_secret = decapsulate(eve.secret_key, alice_result.ciphertext)
        assert alice_result.shared_secret != eve_secret

    def test_tampered_ciphertext_gives_wrong_secret(self):
        """
        ML-KEM est IND-CCA2 : un ciphertext modifié ne lève pas d'exception,
        mais produit un secret incorrect (pas d'oracle d'erreur).
        """
        bob = generate_keypair()
        alice_result = encapsulate(bob.public_key)

        tampered = bytearray(alice_result.ciphertext)
        tampered[0] ^= 0xFF
        tampered[42] ^= 0xAB

        bob_secret = decapsulate(bob.secret_key, bytes(tampered))
        assert alice_result.shared_secret != bob_secret


# ===========================================================================
# 5. KEM RATCHET
# ===========================================================================

class TestKEMRatchet:
    """
    Teste le KEM Ratchet utilisé dans double_ratchet.py.
    Simule plusieurs tours de ratchet entre Alice et Bob.
    """

    def test_kem_ratchet_step_returns_tuple(self):
        bob = generate_keypair()
        result = kem_ratchet_step(bob.public_key)
        assert isinstance(result, tuple)
        assert len(result) == 2

    def test_kem_ratchet_new_public_key_correct_size(self):
        bob = generate_keypair()
        new_pk, _ = kem_ratchet_step(bob.public_key)
        assert len(new_pk) == PK_SIZE

    def test_kem_ratchet_shared_secret_correct_size(self):
        bob = generate_keypair()
        _, secret = kem_ratchet_step(bob.public_key)
        assert len(secret) == SS_SIZE

    def test_two_ratchet_steps_give_different_secrets(self):
        bob = generate_keypair()
        _, s1 = kem_ratchet_step(bob.public_key)
        _, s2 = kem_ratchet_step(bob.public_key)
        assert s1 != s2


# ===========================================================================
# SCRIPT DE DÉMONSTRATION
# ===========================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("  ML-KEM-768 — Protocole QSMP")
    print("=" * 60)

    print("\n Paramètres :")
    for k, v in kem_info().items():
        print(f"   {k:<28} {v}")

    print(f"\nML-KEM-768 disponible : {is_available()}")

    print("\n Bob génère ses clés...")
    bob = generate_keypair()
    print(f"   {bob}")

    print("\n Alice encapsule un secret...")
    alice_result = encapsulate(bob.public_key)
    print(f"   {alice_result}")

    print("\n Bob décapsule...")
    bob_secret = decapsulate(bob.secret_key, alice_result.ciphertext)
    print(f"   Secret Bob : {bob_secret.hex()}")

    match = alice_result.shared_secret == bob_secret
    print(f"\n{' Secrets identiques' if match else ' ERREUR : secrets différents'}")

    print("\n KEM Ratchet — 3 tours...")
    current_bob_pk = bob.public_key
    for i in range(1, 4):
        new_pk, ratchet_secret = kem_ratchet_step(current_bob_pk)
        print(f"   Tour {i} : secret={ratchet_secret.hex()[:32]}…")
        current_bob_pk = new_pk

    print("\n" + "=" * 60)


  ML-KEM-768 — Protocole QSMP

 Paramètres :
   algorithm                    ML-KEM-768
   claimed_nist_level           3
   public_key_bytes             1184
   secret_key_bytes             2400
   ciphertext_bytes             1088
   shared_secret_bytes          32

ML-KEM-768 disponible : True

 Bob génère ses clés...
   KEMKeyPair(pk[1184B]=159cca26f02bfd30…, sk[2400B]=***)

 Alice encapsule un secret...
   KEMEncapsulationResult(ct[1088B]=01f9e9de86ee7c8b…, ss=4627224e31a8111b3a7b8a3f83a11a07339574a202cca77d26928f3bb9a2c665)

 Bob décapsule...
   Secret Bob : 4627224e31a8111b3a7b8a3f83a11a07339574a202cca77d26928f3bb9a2c665

 Secrets identiques

 KEM Ratchet — 3 tours...
   Tour 1 : secret=0d4b4f63ca8c60527a992db7e0c8b3ea…
   Tour 2 : secret=2b847f00701099d9acc728b57bea83ee…
   Tour 3 : secret=8a65e0724f0d5da4b18567ae41ae34cd…

